# 🔔 Akıllı Hatırlatma Sistemi

**Amaç:** Çiftçiye zamanı gelen bakım işlerini otomatik hatırlatan bir sistem
kurmak. Örneğin aşı zamanı, tahmini doğum tarihi, ilaç bekleme süresinin bitişi.

**Yöntem:** Tarih tabanlı kural mantığı. Her hayvanın kayıtlı tarihlerine bakıp,
bugüne göre "yaklaşan" veya "geçmiş" görevleri tespit eder.

**Not:** Bu bölüm yapay zeka değil, tarih/kural tabanlı bir sistemdir; ancak
uygulamanın en pratik ve günlük kullanılan özelliklerinden biridir.

In [ ]:
from datetime import datetime, timedelta

# Bugünün tarihi
bugun = datetime.now()

# Örnek hayvan kayıtları (gerçekte bu veriler veritabanından gelecek)
hayvanlar = [
    {"kupe_no": "TR-001", "ad": "Sarıkız",
     "son_asi": datetime(2025, 2, 1), "asi_periyodu_gun": 180,
     "gebelik_tarihi": datetime(2025, 1, 15),
     "ilac_bekleme_bitis": None},

    {"kupe_no": "TR-002", "ad": "Benekli",
     "son_asi": datetime(2025, 7, 20), "asi_periyodu_gun": 180,
     "gebelik_tarihi": None,
     "ilac_bekleme_bitis": datetime(2025, 8, 5)},

    {"kupe_no": "TR-003", "ad": "Karakaş",
     "son_asi": datetime(2025, 6, 10), "asi_periyodu_gun": 180,
     "gebelik_tarihi": datetime(2024, 12, 1),
     "ilac_bekleme_bitis": None},
]

print("Bugün:", bugun.strftime("%d.%m.%Y"))
print("Kayıtlı hayvan sayısı:", len(hayvanlar))

Bugün: 03.08.2026
Kayıtlı hayvan sayısı: 3


In [ ]:
def hatirlatmalari_getir(hayvan):
    """Bir hayvan için zamanı gelen/yaklaşan hatırlatmaları üretir."""
    hatirlatmalar = []

    # 1. AŞI kontrolü
    if hayvan["son_asi"]:
        sonraki_asi = hayvan["son_asi"] + timedelta(days=hayvan["asi_periyodu_gun"])
        kalan_gun = (sonraki_asi - bugun).days
        if kalan_gun < 0:
            hatirlatmalar.append(f"💉 AŞI GECİKMİŞ! ({-kalan_gun} gün önce yapılmalıydı)")
        elif kalan_gun <= 15:
            hatirlatmalar.append(f"💉 Aşı zamanı yaklaşıyor ({kalan_gun} gün kaldı)")

    # 2. DOĞUM kontrolü (gebelik ~283 gün)
    if hayvan["gebelik_tarihi"]:
        tahmini_dogum = hayvan["gebelik_tarihi"] + timedelta(days=283)
        kalan_gun = (tahmini_dogum - bugun).days
        if kalan_gun < 0:
            hatirlatmalar.append(f"🐄 DOĞUM TARİHİ GEÇMİŞ! (tahmini {-kalan_gun} gün önceydi, kontrol edin)")
        elif kalan_gun <= 15:
            hatirlatmalar.append(f"🐄 Doğum yaklaşıyor ({kalan_gun} gün kaldı) — doğum bölmesini hazırlayın")

    # 3. İLAÇ BEKLEME kontrolü
    if hayvan["ilac_bekleme_bitis"]:
        kalan_gun = (hayvan["ilac_bekleme_bitis"] - bugun).days
        if kalan_gun < 0:
            hatirlatmalar.append(f"💊 İlaç bekleme süresi bitti — süt/et artık kullanılabilir")
        elif kalan_gun >= 0:
            hatirlatmalar.append(f"⛔ İlaç bekleme süresi devam ediyor ({kalan_gun} gün kaldı) — süt/et KULLANMAYIN")

    return hatirlatmalar


# Tüm hayvanlar için hatırlatmaları göster
print("=" * 55)
print(f"📅 HATIRLATMALAR — {bugun.strftime('%d.%m.%Y')}")
print("=" * 55)

for hayvan in hayvanlar:
    mesajlar = hatirlatmalari_getir(hayvan)
    if mesajlar:
        print(f"\n🐮 {hayvan['ad']} ({hayvan['kupe_no']})")
        for m in mesajlar:
            print(f"   {m}")

print("\n" + "=" * 55)

📅 HATIRLATMALAR — 03.08.2026

🐮 Sarıkız (TR-001)
   💉 AŞI GECİKMİŞ! (369 gün önce yapılmalıydı)
   🐄 DOĞUM TARİHİ GEÇMİŞ! (tahmini 283 gün önceydi, kontrol edin)

🐮 Benekli (TR-002)
   💉 AŞI GECİKMİŞ! (200 gün önce yapılmalıydı)
   💊 İlaç bekleme süresi bitti — süt/et artık kullanılabilir

🐮 Karakaş (TR-003)
   💉 AŞI GECİKMİŞ! (240 gün önce yapılmalıydı)
   🐄 DOĞUM TARİHİ GEÇMİŞ! (tahmini 328 gün önceydi, kontrol edin)



In [ ]:
# Yaklaşan durumu test etmek için: bugüne yakın tarihli bir hayvan
test_hayvan = {
    "kupe_no": "TR-004", "ad": "Yıldız",
    "son_asi": bugun - timedelta(days=170),   # 170 gün önce aşı (periyot 180, yani 10 gün kaldı)
    "asi_periyodu_gun": 180,
    "gebelik_tarihi": bugun - timedelta(days=275),  # 275 gün önce gebe (283'e 8 gün kaldı)
    "ilac_bekleme_bitis": None
}

print(f"🐮 {test_hayvan['ad']} ({test_hayvan['kupe_no']})")
for m in hatirlatmalari_getir(test_hayvan):
    print(f"   {m}")

🐮 Yıldız (TR-004)
   💉 Aşı zamanı yaklaşıyor (10 gün kaldı)
   🐄 Doğum yaklaşıyor (8 gün kaldı) — doğum bölmesini hazırlayın


## 🎯 Kapanış — Akıllı Hatırlatma Sistemi

Bu notebook'ta, çiftçiye zamanı gelen bakım işlerini otomatik hatırlatan tarih
tabanlı bir sistem kurduk.

**Yapılanlar**
- Her hayvan için aşı, doğum ve ilaç bekleme tarihlerini takip etme
- `datetime` ve `timedelta` ile "kaç gün kaldı / kaç gün geçti" hesabı
- Üç tür hatırlatma: aşı zamanı (periyoda göre), tahmini doğum (283 günlük
  gebelik hesabı), ilaç bekleme süresi (gıda güvenliği için)
- Hem geçmiş/geciken hem de yaklaşan görevleri tespit etme

**Sonuç**
Sistem, her hayvanın kayıtlı tarihlerine bakarak "aşı gecikmiş", "doğum yaklaşıyor
(8 gün kaldı)", "ilaç bekleme süresi devam ediyor" gibi net hatırlatmalar üretiyor.
Bu, uygulamanın günlük kullanılan, pratik özelliklerinden biridir.

**Not:** Şimdilik örnek verilerle çalışıyor; gerçek uygulamada bu tarihler
veritabanından (sağlık kaydı ve gebelik tablolarından) gelecektir.

şimdi hatırlatmalara güzel bi şey eklemeye karar verdim bu colabdan devam edicez.Uygulamaya giriş yaptıgımızda sağ üstte rozet renklerine göre hatırlatmalar vermek istedim sabah raporu gibi bi şey olacak.Bu sayede çiftçi asistanı varmış gibi o gün yapılması gereken ,yaklaşan ,veya sorun yok gibi rozetleri görebilecek ve günlük iş akışını ona göre yapacak.Hatırlatmalar bu colab dosyamda oldugu için burdan devam etmek istedim.


In [4]:
def oncelik_belirle(hayvan):
    """Bir hayvanın hatırlatmalarını ACİL / YAKLAŞAN olarak sınıflar."""
    acil = []
    yaklasan = []

    if hayvan["son_asi"]:
        kalan = (hayvan["son_asi"] + timedelta(days=hayvan["asi_periyodu_gun"]) - bugun).days
        if kalan < 0:
            acil.append(f"💉 {hayvan['ad']}: Aşı gecikmiş ({-kalan} gün)")
        elif kalan <= 15:
            yaklasan.append(f"💉 {hayvan['ad']}: Aşıya {kalan} gün kaldı")

    if hayvan["gebelik_tarihi"]:
        kalan = (hayvan["gebelik_tarihi"] + timedelta(days=283) - bugun).days
        if kalan < 0:
            acil.append(f"🐄 {hayvan['ad']}: Doğum tarihi geçmiş, kontrol edin!")
        elif kalan <= 3:
            acil.append(f"🐄 {hayvan['ad']}: Doğuma {kalan} gün! Bölme hazır olsun")
        elif kalan <= 15:
            yaklasan.append(f"🐄 {hayvan['ad']}: Doğuma {kalan} gün kaldı")

    if hayvan["ilac_bekleme_bitis"]:
        kalan = (hayvan["ilac_bekleme_bitis"] - bugun).days
        if kalan >= 0:
            acil.append(f"⛔ {hayvan['ad']}: İlaç beklemesi sürüyor ({kalan} gün) — süt/et KULLANMAYIN")

    return acil, yaklasan


def bildirim_rozeti():
    """Köşedeki uyarı rozetinin durumunu hesaplar (mobil arayüz için beyin)."""
    tum_acil = []
    tum_yaklasan = []
    for hayvan in hayvanlar:
        a, y = oncelik_belirle(hayvan)
        tum_acil += a
        tum_yaklasan += y

    if tum_acil:
        renk = "🔴 KIRMIZI"
        sayi = len(tum_acil)
    elif tum_yaklasan:
        renk = "🟡 SARI"
        sayi = len(tum_yaklasan)
    else:
        renk = "🟢 YOK"
        sayi = 0

    return renk, sayi, tum_acil, tum_yaklasan


renk, sayi, acil, yaklasan = bildirim_rozeti()
print("┌" + "─" * 40 + "┐")
print(f"│  ANA EKRAN ROZETİ:  {renk}  ({sayi})")
print("└" + "─" * 40 + "┘")
print()

print("=" * 45)
print("📋 GÜNLÜK RAPOR (rozete tıklanınca)")
print("=" * 45)

if acil:
    print("\n🔴 ACİL:")
    for a in acil:
        print(f"   {a}")

if yaklasan:
    print("\n🟡 YAKLAŞAN:")
    for y in yaklasan:
        print(f"   {y}")

if not acil and not yaklasan:
    print("\n🟢 Acil veya yaklaşan bir durum yok. Her şey yolunda!")
print("\n" + "=" * 45)

┌────────────────────────────────────────┐
│  ANA EKRAN ROZETİ:  🔴 KIRMIZI  (5)
└────────────────────────────────────────┘

📋 GÜNLÜK RAPOR (rozete tıklanınca)

🔴 ACİL:
   💉 Sarıkız: Aşı gecikmiş (369 gün)
   🐄 Sarıkız: Doğum tarihi geçmiş, kontrol edin!
   💉 Benekli: Aşı gecikmiş (200 gün)
   💉 Karakaş: Aşı gecikmiş (240 gün)
   🐄 Karakaş: Doğum tarihi geçmiş, kontrol edin!



In [5]:
# Yaklaşan (sarı) durumu da test etmek için Yıldız'ı ekle
hayvanlar.append({
    "kupe_no": "TR-004", "ad": "Yıldız",
    "son_asi": bugun - timedelta(days=170),
    "asi_periyodu_gun": 180,
    "gebelik_tarihi": bugun - timedelta(days=275),
    "ilac_bekleme_bitis": None
})

# Rozeti tekrar hesapla
renk, sayi, acil, yaklasan = bildirim_rozeti()
print(f"ROZET: {renk} ({sayi} acil)")
print()
if acil:
    print("🔴 ACİL:")
    for a in acil: print(f"   {a}")
if yaklasan:
    print("\n🟡 YAKLAŞAN:")
    for y in yaklasan: print(f"   {y}")

ROZET: 🔴 KIRMIZI (5 acil)

🔴 ACİL:
   💉 Sarıkız: Aşı gecikmiş (369 gün)
   🐄 Sarıkız: Doğum tarihi geçmiş, kontrol edin!
   💉 Benekli: Aşı gecikmiş (200 gün)
   💉 Karakaş: Aşı gecikmiş (240 gün)
   🐄 Karakaş: Doğum tarihi geçmiş, kontrol edin!

🟡 YAKLAŞAN:
   💉 Yıldız: Aşıya 10 gün kaldı
   🐄 Yıldız: Doğuma 8 gün kaldı


## 🎯 Kapanış— Öncelikli Bildirim Rozeti Sistemi

Bu bölümde, hatırlatmaları önem sırasına göre sınıflayan ve ana ekranda tek bakışta
görülebilecek bir uyarı rozeti sistemi kurduk.

**Yapılanlar**
- Hatırlatmaları ACİL (🔴) ve YAKLAŞAN (🟡) olarak önceliklendirme
- Acil sayılanlar: gecikmiş aşı/doğum, doğuma 3 günden az, süren ilaç beklemesi
  (gıda güvenliği)
- Yaklaşan sayılanlar: 15 gün içindeki aşı ve doğumlar
- Ana ekran rozetinin rengini ve sayısını otomatik belirleme (acil varsa kırmızı,
  sadece yaklaşan varsa sarı, hiçbiri yoksa rozet yok)
- Rozete tıklanınca açılan, önceliğe göre sıralı detay raporu

**Tasarım mantığı**
Çiftçinin her açılışta uzun bir panel okumasına gerek yok. Köşedeki kırmızı rozet,
acil bir durum olup olmadığını tek bakışta gösterir; çiftçi vakti olunca tıklayıp
detayı görür. Bu, kullanıcının zamanını gözeten pratik bir bildirim tasarımıdır.

**Not:** Rozetin görsel hali mobil arayüzde olacaktır; bu bölümde onun karar
mekanizması (hangi durum hangi renk/öncelik) kurulmuştur.